# MobileNetV2 Diagnostic & Interpretability Analysis: Merged Dataset (Model C)
---
Full diagnostic suite for **Model C** (jointly trained on PlatesMania + External datasets across 1,235 Make/Model classes and 35 makes), covering training curves, per-domain test metrics, model & make confusion matrices, latent space projections (t-SNE & PCA), confidence calibration, mixed benchmark comparison, and analytical Grad-CAM.


## 1. Setup & Context Initialization


In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

import output_analysis_merged as oam

RUN_DIR = oam.DEFAULT_RUN_DIR
print(f"Run Directory: {RUN_DIR}")
print(f"Context Model: {oam.context.model_path}")
print(f"Classes:       {oam.context.num_classes}")


## 2. Training Curves & Validation History


In [ ]:
train_curves = RUN_DIR / "plots" / "training_curves.png"
if train_curves.exists():
    display(Image(str(train_curves)))

hist_csv = RUN_DIR / "reports" / "training_history.csv"
if hist_csv.exists():
    display(pd.read_csv(hist_csv).tail(5))


## 3. PlatesMania Test Split Diagnostics (`diagnostics_platesmania/`)


In [ ]:
pm_diag_dir = RUN_DIR / "diagnostics_platesmania"
pm_metrics_file = pm_diag_dir / "reports" / "test_metrics.json"
if pm_metrics_file.exists():
    with open(pm_metrics_file) as f:
        print("=== PlatesMania Full Test Metrics ===")
        print(json.dumps(json.load(f), indent=2))

for plot_name in ["confusion_matrix.png", "confusion_matrix_make.png", "latent_space_tsne_pca.png", "calibration_curve.png"]:
    p = pm_diag_dir / "plots" / plot_name
    if p.exists():
        display(Image(str(p)))


## 4. External Test Split Diagnostics (`diagnostics_external/`)


In [ ]:
ext_diag_dir = RUN_DIR / "diagnostics_external"
ext_metrics_file = ext_diag_dir / "reports" / "test_metrics.json"
if ext_metrics_file.exists():
    with open(ext_metrics_file) as f:
        print("=== External Full Test Metrics ===")
        print(json.dumps(json.load(f), indent=2))

for plot_name in ["confusion_matrix.png", "confusion_matrix_make.png", "latent_space_tsne_pca.png", "calibration_curve.png"]:
    p = ext_diag_dir / "plots" / plot_name
    if p.exists():
        display(Image(str(p)))


## 5. Mixed Benchmark Summary (`benchmark/`)


In [ ]:
bench_dir = RUN_DIR / "benchmark"
summary_csv = bench_dir / "mixed_benchmark_summary.csv"
if summary_csv.exists():
    display(pd.read_csv(summary_csv))

shared_csv = bench_dir / "shared_class_summary.csv"
if shared_csv.exists():
    display(pd.read_csv(shared_csv))

comp_plot = bench_dir / "cross_domain_comparison.png"
if comp_plot.exists():
    display(Image(str(comp_plot)))


## 6. Analytical Grad-CAM (PlatesMania & External Test Samples)
Computes analytical class activation maps from the ONNX `class_maps` output across random test samples from both domains.


In [ ]:
model = oam.MobileNetV2Evaluator(num_classes=oam.context.num_classes, checkpoint_path=oam.context.model_path)

for domain in ["platesmania", "external"]:
    ctx = oam.configure_domain(domain, RUN_DIR)
    test_csv = Path(ctx.test_csv_path)
    if not test_csv.exists():
        print(f"[{domain}] Test CSV not found: {test_csv}")
        continue
    df_test = pd.read_csv(test_csv)
    if "split" in df_test.columns:
        df_test = df_test[df_test["split"] == "test"].reset_index(drop=True)
    col = "image_path" if "image_path" in df_test.columns else "image_rel_path"
    sample = df_test.sample(n=min(3, len(df_test)), random_state=42)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"Model C Grad-CAM — {domain.capitalize()} Test Split", fontsize=13, fontweight="bold")
    for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
        ax.axis("off")
        p = row[col]
        if ctx.base_img_dir and not os.path.isabs(p):
            p = str(Path(ctx.base_img_dir) / p)
        bbox = None
        if "bbox_x1" in row and int(row["bbox_x1"]) >= 0:
            bbox = (int(row["bbox_x1"]), int(row["bbox_y1"]), int(row["bbox_x2"]), int(row["bbox_y2"]))
        tensor, img = oam.load_and_preprocess_image(p, ctx.img_size, ctx.crop_top_pct, ctx.crop_bottom_pct, bbox=bbox)
        pred = int(model(tensor, return_embeddings=False, return_cam=True).argmax(dim=1).item())
        cam = oam.compute_gradcam(model, tensor, pred)
        ax.imshow(oam.overlay_cam_on_image(img, cam))
        ax.set_title(f"true: {ctx.idx_to_class[int(row['label'])]}\npred: {ctx.idx_to_class[pred]}", fontsize=10)
    plt.tight_layout()
    plt.show()
